# IT Architect Assistant — Retrieval system on GCP

This notebook builds the *retrieval* half of the RAG system: it turns the referential of design principles stored in **Google Drive** into a searchable index (a **corpus**) on Google Cloud, then checks that questions bring back the right passages.

```
Google Drive folder ──► import ──► chunking ──► embeddings ──► corpus (index)
                                                                    ▲
                                              question ──► retrieval ┘──► top matching passages
```

**How to use it**

1. Edit `config.py` (same folder as this notebook). Every setting lives there; you should not need to change any code cell.
2. Run the cells from top to bottom. Each step ends with a **Checkpoint** telling you what you should see before moving on.
3. Step 5 contains one manual action (sharing the Drive folder). Do not skip it.

**Before you start, you need**

- A GCP project with billing enabled, and the *Vertex AI User* role (`roles/aiplatform.user`) on it. Enabling the API in step 2 also needs *Service Usage Admin* or Owner.
- Edit rights on the Drive folder that holds the referential, so you can share it.
- A Python environment: Colab Enterprise or Vertex AI Workbench (recommended, already authenticated), or local Jupyter with the `gcloud` CLI installed.

## Step 0 — Install the SDK

`google-cloud-agentplatform` is Google's current Python SDK for RAG Engine (it replaces the deprecated `vertexai.rag` module).

**Checkpoint:** the cell finishes without a red error. If it asks you to restart the kernel, do so, then continue from step 1.

In [ ]:
import sys

!uv pip install --python "{sys.executable}" --upgrade google-cloud-agentplatform google-cloud-storage pandas

## Step 1 — Load and check the configuration

This cell reads `config.py` and stops immediately if a `TODO` value was left unchanged, so you don't discover it ten minutes later.

**Checkpoint:** a summary table of your settings, and the line `Configuration looks good`.

In [ ]:
import importlib
import json
import subprocess

import pandas as pd

import config
import test_queries
importlib.reload(config)  # picks up edits to config.py without restarting the kernel
importlib.reload(test_queries)  # same for test_queries.py

problems = []
if config.PROJECT_ID.startswith("your-"):
    problems.append("PROJECT_ID is still the placeholder.")
folder_ids = [i for i in config.DRIVE_FOLDER_IDS if i and not i.startswith("your-")]
file_ids = [i for i in config.DRIVE_FILE_IDS if i and not i.startswith("your-")]
if not folder_ids and not file_ids:
    problems.append("Add at least one ID in DRIVE_FOLDER_IDS or DRIVE_FILE_IDS.")
if config.CHUNK_OVERLAP >= config.CHUNK_SIZE:
    problems.append("CHUNK_OVERLAP must be smaller than CHUNK_SIZE.")
if not test_queries.TEST_QUERIES:
    problems.append("TEST_QUERIES is empty (see test_queries.py).")

display(pd.DataFrame(
    [
        ("Project", config.PROJECT_ID),
        ("Region", config.LOCATION),
        ("Corpus name", config.CORPUS_DISPLAY_NAME),
        ("Embedding model", config.EMBEDDING_MODEL),
        ("Drive folders", len(folder_ids)),
        ("Drive files", len(file_ids)),
        ("Chunk size / overlap", f"{config.CHUNK_SIZE} / {config.CHUNK_OVERLAP}"),
        ("Top K", config.TOP_K),
        ("Distance threshold", config.VECTOR_DISTANCE_THRESHOLD),
    ],
    columns=["Setting", "Value"],
))

if problems:
    raise ValueError("Fix config.py, then re-run this cell:\n- " + "\n- ".join(problems))
print("Configuration looks good.")

## Step 2 — Authenticate locally and enable the API

This notebook is configured to run from your local machine.

Before running this cell, make sure you have authenticated with Google Cloud locally:

```bash
gcloud auth application-default login
gcloud config set project YOUR_PROJECT_ID
```

The cell then verifies your local credentials, enables the required API, and reads your project number.

**Checkpoint:** your active account and project number are printed, and no permission error appears.

In [ ]:
import shutil
import subprocess
from pathlib import Path

import config


def gcloud(*args: str) -> str:
    """Run a local gcloud command and return its output, raising a readable error on failure."""
    if shutil.which("gcloud") is None:
        raise RuntimeError(
            "The gcloud CLI is not installed or is not on PATH. "
            "Install it, then run `gcloud auth application-default login` in a terminal."
        )

    result = subprocess.run(["gcloud", *args], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"gcloud {' '.join(args)} failed: {result.stderr.strip()}")
    return result.stdout.strip()


active_account = gcloud("auth", "list", "--filter=status:ACTIVE", "--format=value(account)")
if not active_account:
    raise RuntimeError(
        "No active local gcloud account found. Run `gcloud auth login` and "
        "`gcloud auth application-default login` in a terminal, then re-run this cell."
    )
print("Active gcloud account:", active_account)

adc_path = Path.home() / ".config" / "gcloud" / "application_default_credentials.json"
if not adc_path.exists():
    raise RuntimeError(
        "Application Default Credentials were not found. Run "
        "`gcloud auth application-default login` in a terminal, then re-run this cell."
    )
print("Application Default Credentials found:", adc_path)

gcloud("config", "set", "project", config.PROJECT_ID)
gcloud("services", "enable", *config.REQUIRED_APIS, "--project", config.PROJECT_ID)
print("APIs enabled:", ", ".join(config.REQUIRED_APIS))

PROJECT_NUMBER = gcloud(
    "projects", "describe", config.PROJECT_ID, "--format=value(projectNumber)"
)
print("Project number:", PROJECT_NUMBER)

## Step 3 — Connect to RAG Engine

We create one `client` object. Every later call goes through `client.rag`.

**Checkpoint:** the line `Client ready` with your project and region.

In [ ]:
import agentplatform
from agentplatform import types
from google.genai import types as genai_types

client = agentplatform.Client(project=config.PROJECT_ID, location=config.LOCATION)
print(f"Client ready: project={config.PROJECT_ID}, region={config.LOCATION}")

## Step 4 — Create the corpus (the empty index)

A **corpus** is the container that will hold the chunks of your referential and their embeddings. At this point it is empty.

The cell is safe to re-run: if a corpus with the name from `config.py` already exists, it is reused instead of creating a duplicate.

**Checkpoint:** a line starting with `Corpus:` followed by a resource name such as `projects/.../locations/.../ragCorpora/1234567890`. The first creation can take a few minutes.

In [ ]:
def find_corpus(display_name: str):
    """Return the existing corpus with this display name, or None."""
    page_token = None
    while True:
        page = client.rag.list_corpora(
            config={"page_token": page_token} if page_token else None
        )
        for corpus in page.rag_corpora or []:
            if corpus.display_name == display_name:
                return corpus
        page_token = page.next_page_token
        if not page_token:
            return None


rag_corpus = find_corpus(config.CORPUS_DISPLAY_NAME)

if rag_corpus:
    print("Reusing the existing corpus.")
else:
    print("Creating the corpus (this can take a few minutes)...")
    rag_corpus = client.rag.create_corpus(
        rag_corpus=types.RagCorpus(
            display_name=config.CORPUS_DISPLAY_NAME,
            description=config.CORPUS_DESCRIPTION,
            rag_vector_db_config=types.RagVectorDbConfig(
                rag_embedding_model_config=types.RagEmbeddingModelConfig(
                    vertex_prediction_endpoint=types.RagEmbeddingModelConfigVertexPredictionEndpoint(
                        endpoint=config.EMBEDDING_MODEL
                    )
                )
            ),
        )
    )
    print("Corpus created.")

CORPUS_NAME = rag_corpus.name
print("Corpus:", CORPUS_NAME)

## Step 5 — Give RAG Engine access to the Drive folder (manual action)

RAG Engine reads Drive through a Google-managed robot account, the **RAG Data Service Agent**. It can only see what you share with it.

Run the cell below to print its e-mail address, then:

1. Open the Drive folder (or file) that holds the referential.
2. Click **Share**.
3. Paste the address, choose the **Viewer** role, untick *Notify people*, and confirm.

Two things to know:

- If the share is missing, the import does **not** fail with an error: it simply imports 0 files. Step 6 checks for that.
- The folder should sit in *My Drive* or in a shared drive. Content that only appears under *Shared with me* is often not picked up.

To double-check the address: Cloud console → **IAM** → tick **Include Google-provided role grants** → look for the *RAG Data Service Agent*.

**Checkpoint:** the service agent appears in the folder's sharing list as *Viewer*.

In [ ]:
RAG_SERVICE_AGENT = f"service-{PROJECT_NUMBER}@gcp-sa-vertex-rag.iam.gserviceaccount.com"

print("Share your Drive folder(s) with this address, as Viewer:\n")
print("   ", RAG_SERVICE_AGENT, "\n")
for folder_id in folder_ids:
    print("Folder:", f"https://drive.google.com/drive/folders/{folder_id}")
for file_id in file_ids:
    print("File:  ", f"https://drive.google.com/file/d/{file_id}")

## Step 6 — Import the referential from Drive

This is the indexing pipeline. For each document, RAG Engine:

1. reads it from Drive,
2. cuts it into **chunks** of `CHUNK_SIZE` tokens that overlap by `CHUNK_OVERLAP` tokens (the overlap avoids cutting a rule in half),
3. computes an **embedding** for each chunk,
4. stores chunk + embedding in the corpus.

Re-running the cell is safe: files already imported and unchanged are skipped.

**Checkpoint:** `imported` is greater than 0 on the first run (on later runs `skipped` is greater than 0 instead), and `failed` is 0. Expect a few minutes for a large folder.

In [ ]:
resource_ids = [
    types.GoogleDriveSourceResourceId(
        resource_id=folder_id, resource_type="RESOURCE_TYPE_FOLDER"
    )
    for folder_id in folder_ids
] + [
    types.GoogleDriveSourceResourceId(
        resource_id=file_id, resource_type="RESOURCE_TYPE_FILE"
    )
    for file_id in file_ids
]

print(f"Importing from {len(resource_ids)} Drive location(s)... please wait.")
import_response = client.rag.import_files(
    name=CORPUS_NAME,
    import_config=types.ImportRagFilesConfig(
        google_drive_source=types.GoogleDriveSource(resource_ids=resource_ids),
        rag_file_transformation_config=types.RagFileTransformationConfig(
            rag_file_chunking_config=types.RagFileChunkingConfig(
                fixed_length_chunking=types.RagFileChunkingConfigFixedLengthChunking(
                    chunk_size=config.CHUNK_SIZE,
                    chunk_overlap=config.CHUNK_OVERLAP,
                )
            )
        ),
        max_embedding_requests_per_min=config.MAX_EMBEDDING_REQUESTS_PER_MIN,
    ),
)

imported = int(import_response.imported_rag_files_count or 0)
skipped = int(import_response.skipped_rag_files_count or 0)
failed = int(import_response.failed_rag_files_count or 0)
print(f"imported={imported}  skipped={skipped}  failed={failed}")

if imported + skipped == 0:
    print(
        "\nNothing was imported. Almost always this means the Drive folder is not "
        "shared with the service agent from step 5 (as Viewer), or the ID in "
        "config.py is wrong."
    )
if failed:
    print("\nSome files failed. Unsupported formats and oversized files are the usual causes.")

## Step 7 — Check what is in the corpus

Lists every file now indexed, with its status. Compare the list with your Drive folder.

**Checkpoint:** the number of rows matches the number of documents you expected, and every `state` is `ACTIVE`.

In [ ]:
def list_all_files(corpus_name: str) -> list:
    files, page_token = [], None
    while True:
        cfg = {"page_size": 100}
        if page_token:
            cfg["page_token"] = page_token
        page = client.rag.list_files(name=corpus_name, config=cfg)
        files.extend(page.rag_files or [])
        page_token = page.next_page_token
        if not page_token:
            return files


rag_files = list_all_files(CORPUS_NAME)
files_df = pd.DataFrame(
    [
        {
            "file": f.display_name,
            "state": getattr(getattr(f.file_status, "state", None), "value", None),
            "size_bytes": f.size_bytes,
            "updated": f.update_time,
        }
        for f in rag_files
    ]
)
print(f"{len(rag_files)} file(s) in the corpus.")
display(files_df)

## Step 8 — Build the retrieval function

`retrieve(question)` is the function the chatbot will call later. It embeds the question, finds the closest chunks in the corpus, and returns them as a table with their source document.

How to read the result:

- **text**: the passage found.
- **source**: the document it comes from (this is what the chatbot will cite).
- **distance**: how far the passage is from the question. **Smaller is closer.**

**Checkpoint:** the cell runs without output (it only defines the function).

In [ ]:
def retrieve(question: str, top_k: int | None = None) -> pd.DataFrame:
    """Return the passages of the referential that best match the question."""
    threshold = config.VECTOR_DISTANCE_THRESHOLD
    response = client.rag.retrieve_contexts(
        vertex_rag_store=genai_types.VertexRagStore(
            rag_resources=[genai_types.VertexRagStoreRagResource(rag_corpus=CORPUS_NAME)]
        ),
        query=types.RagQuery(
            text=question,
            rag_retrieval_config=genai_types.RagRetrievalConfig(
                top_k=top_k or config.TOP_K,
                filter=(
                    genai_types.RagRetrievalConfigFilter(vector_distance_threshold=threshold)
                    if threshold is not None
                    else None
                ),
            ),
        ),
    )
    contexts = (response.contexts.contexts if response.contexts else None) or []
    return pd.DataFrame(
        [
            {
                "rank": rank,
                "distance": round(c.distance, 3) if c.distance is not None else None,
                "source": c.source_display_name,
                "text": (c.text or "").strip(),
                "uri": c.source_uri,
            }
            for rank, c in enumerate(contexts, start=1)
        ],
        columns=["rank", "distance", "source", "text", "uri"],
    )

## Step 9 — Test retrieval with real questions

Runs every question from `TEST_QUERIES` in `test_queries.py`. This is the most important check of the notebook: read the passages and ask yourself *"would an architect accept this as the right reference?"*

**Checkpoint:** for each question, the first results come from the document you expected.

If the results are poor:

| Symptom | What to try in `config.py` |
|---|---|
| No result at all | Set `VECTOR_DISTANCE_THRESHOLD = None` |
| Right document, but the passage is cut off | Increase `CHUNK_SIZE` (e.g. 1024), then re-import into a **new** corpus name |
| Passages too vague or mixed topics | Decrease `CHUNK_SIZE` (e.g. 256), same procedure |
| French documents, poor matches | Use the multilingual `EMBEDDING_MODEL`, with a new corpus name |

In [ ]:
import html

from IPython.display import HTML, display

PREVIEW_CHARS = 400  # passages longer than this are collapsed; click to read the full text


def show_results(results: pd.DataFrame) -> None:
    """Display each passage in full: short ones inline, long ones in an expandable block."""
    cards = []
    for row in results.itertuples():
        text = html.escape(row.text)
        body = (
            f"<details><summary style='cursor:pointer'>{html.escape(row.text[:PREVIEW_CHARS])}... "
            f"<b>[click to read the full passage - {len(row.text)} characters]</b></summary>"
            f"<div style='white-space:pre-wrap;max-height:500px;overflow:auto;margin-top:6px;"
            f"padding:8px;border:1px solid #8884;border-radius:4px'>{text}</div></details>"
            if len(row.text) > PREVIEW_CHARS
            else f"<div style='white-space:pre-wrap'>{text}</div>"
        )
        cards.append(
            "<div style='margin:8px 0;padding:10px;border:1px solid #8886;border-radius:6px'>"
            f"<div><b>#{row.rank}</b> &nbsp; distance: <b>{row.distance}</b> &nbsp; "
            f"source: <b>{html.escape(str(row.source))}</b></div>"
            f"<div style='margin-top:6px'>{body}</div></div>"
        )
    display(HTML("".join(cards)))

import importlib
import test_queries
importlib.reload(test_queries)

for question in test_queries.TEST_QUERIES:
    print("\n" + "=" * 100)
    print("QUESTION:", question)
    results = retrieve(question)
    if results.empty:
        print("No passage returned.")
    else:
        show_results(results)


## Step 10 — Save the corpus reference for the chatbot

Writes the corpus resource name to a small JSON file. The chatbot (next stage of the project) only needs this file to plug into the index you just built.

**Checkpoint:** the file named in `CORPUS_INFO_FILE` appears next to the notebook.

In [ ]:
corpus_info = {
    "project_id": config.PROJECT_ID,
    "location": config.LOCATION,
    "corpus_name": CORPUS_NAME,
    "corpus_display_name": config.CORPUS_DISPLAY_NAME,
    "embedding_model": config.EMBEDDING_MODEL,
    "file_count": len(rag_files),
}
with open(config.CORPUS_INFO_FILE, "w", encoding="utf-8") as fh:
    json.dump(corpus_info, fh, indent=2)

print(f"Saved to {config.CORPUS_INFO_FILE}:")
print(json.dumps(corpus_info, indent=2))

## Keeping the index up to date

The import is a snapshot: documents added or edited in Drive afterwards are **not** picked up automatically. To refresh, re-run step 6 (unchanged files are skipped), then steps 7 and 9.

## Clean-up (optional)

A corpus keeps costing money while it exists. This cell deletes the corpus and everything indexed in it. It does nothing unless `ALLOW_DELETE_CORPUS = True` in `config.py`. Your Drive documents are never touched.

In [ ]:
importlib.reload(config)

if config.ALLOW_DELETE_CORPUS:
    client.rag.delete_corpus(name=CORPUS_NAME)
    print("Corpus deleted:", CORPUS_NAME)
else:
    print("Nothing deleted. Set ALLOW_DELETE_CORPUS = True in config.py to enable this cell.")